In [40]:
import json
from pathlib import Path

# Load the vfcs.json file
json_path = Path("./vfcs.json")
with open(json_path, 'r') as f:
    vfcs_data = json.load(f)

print(f"Loaded {len(vfcs_data)} records from vfcs.json")

Loaded 600 records from vfcs.json


In [41]:
number_of_commit = 0
number_of_pr = 0

for datapoint in vfcs_data:
    if datapoint['commit_type'] == 'commit':
        number_of_commit += 1
    elif datapoint['commit_type'] == 'pull':
        number_of_pr += 1

print(f"Number of commits: {number_of_commit}, number of pull requests: {number_of_pr}")

Number of commits: 206, number of pull requests: 13


In [42]:
import json
from pathlib import Path

# Load the vfcs.json file
json_path = Path("./vfcs.filtered.json")
with open(json_path, 'r') as f:
    vfcs_data = json.load(f)

print(f"Loaded {len(vfcs_data)} records from vfcs.filtered.json")

Loaded 190 records from vfcs.filtered.json


In [43]:
number_of_commit = 0
number_of_pr = 0

for datapoint in vfcs_data:
    if datapoint['commit_type'] == 'commit':
        number_of_commit += 1
    elif datapoint['commit_type'] == 'pull':
        number_of_pr += 1

print(f"Number of commits: {number_of_commit}, number of pull requests: {number_of_pr}")

Number of commits: 179, number of pull requests: 11


In [44]:
from statistics import mean, pstdev, quantiles


def calculate_min_max_mean_std(values):
    if not values:
        raise ValueError('values must not be empty')

    min_value = min(values)
    max_value = max(values)
    mean_value = mean(values)
    std_value = pstdev(values) if len(values) > 1 else 0.0
    quartiles = quantiles(values, n=4, method='inclusive') if len(values) > 1 else [values[0], values[0], values[0]]
    q1_value = quartiles[0]
    q2_value = quartiles[1] if len(values) > 1 else values[0]
    q3_value = quartiles[2]

    return {
        'min': min_value,
        'max': max_value,
        'mean': round(mean_value, 2),
        'std': round(std_value, 2),
        'q1': round(q1_value, 2),
        'q2': round(q2_value, 2),
        'q3': round(q3_value, 2),
    }

In [45]:
commit_messages = [datapoint.get('commit_message', '') for datapoint in vfcs_data]
print('Number of commit messages:', len(commit_messages))
calculate_min_max_mean_std([len(commit_message) for commit_message in commit_messages])

Number of commit messages: 190


{'min': 8,
 'max': 3181,
 'mean': 116.66,
 'std': 278.07,
 'q1': 31.0,
 'q2': 54.0,
 'q3': 98.0}

In [46]:
code_changes = [datapoint.get('source_code_changes', []) for datapoint in vfcs_data]
print('Number of source code changes:', len(commit_messages))
calculate_min_max_mean_std([len(code_change) for code_change in code_changes])

Number of source code changes: 190


{'min': 1,
 'max': 40,
 'mean': 1.86,
 'std': 3.41,
 'q1': 1.0,
 'q2': 1.0,
 'q3': 1.0}

In [47]:
code_changes = [datapoint.get('all_code_changes', []) for datapoint in vfcs_data]
print('Number of all code changes:', len(commit_messages))
calculate_min_max_mean_std([len(code_change) for code_change in code_changes])

Number of all code changes: 190


{'min': 1,
 'max': 49,
 'mean': 3.57,
 'std': 4.97,
 'q1': 1.0,
 'q2': 2.0,
 'q3': 3.0}

In [48]:
VULN_TYPE_LABELS = {
    'path-traversal': 'Path Traversal',
    'prototype-pollution': 'Prototype Pollution',
    'command-injection': 'Command Injection',
    'code-injection': 'Code Injection',
    'redos': 'ReDoS',
}


def classify_source(ids):
    for id_val in ids:
        if id_val.startswith('GHSA-'):
            return 'GHSA'
    return 'Snyk'


def print_distribution_table(records, title):
    counts = {vtype: {'GHSA': 0, 'Snyk': 0} for vtype in VULN_TYPE_LABELS}

    for record in records:
        vtype = record.get('vulnerability_type', '')
        ids = record.get('ids', [])
        source = classify_source(ids)
        if vtype in counts:
            counts[vtype][source] += 1

    col_w = 25
    print(f'Distribution of vulnerability types in {title}')
    print(f'{"Vulnerability Class":<{col_w}} {"GHSA":>6} {"Snyk":>6} {"Total":>7}')
    print('-' * (col_w + 22))
    total_ghsa = total_snyk = 0
    for vtype, label in VULN_TYPE_LABELS.items():
        g, s = counts[vtype]['GHSA'], counts[vtype]['Snyk']
        total_ghsa += g; total_snyk += s
        print(f'{label:<{col_w}} {g:>6} {s:>6} {g + s:>7}')
    print('-' * (col_w + 22))
    grand = total_ghsa + total_snyk
    print(f'{"Total":<{col_w}} {total_ghsa:>6} {total_snyk:>6} {grand:>7}')


testbed_data = json.loads(Path('./vfcs.testbed.json').read_text())
print_distribution_table(testbed_data, 'vfcs.testbed.json')
print()
filtered_data = json.loads(Path('./vfcs.filtered.json').read_text())
print_distribution_table(filtered_data, 'vfcs.filtered.json')

Distribution of vulnerability types in vfcs.testbed.json
Vulnerability Class         GHSA   Snyk   Total
-----------------------------------------------
Path Traversal                85     82     167
Prototype Pollution          136     44     180
Command Injection             66     24      90
Code Injection                26     10      36
ReDoS                         59     27      86
-----------------------------------------------
Total                        372    187     559

Distribution of vulnerability types in vfcs.filtered.json
Vulnerability Class         GHSA   Snyk   Total
-----------------------------------------------
Path Traversal                 8      2      10
Prototype Pollution           61     20      81
Command Injection             24      4      28
Code Injection                 9      4      13
ReDoS                         40     18      58
-----------------------------------------------
Total                        142     48     190


In [49]:
# Analyse test vs non-test file changes in vfcs.filtered.json
# A file change is considered "test-related" when "test" appears anywhere in its filename path

def is_test_file(filename: str) -> bool:
    return 'test' in filename.lower()

records_with_test = 0    # VFC records that include at least one test file change
records_without_test = 0 # VFC records with no test file changes at all

for datapoint in filtered_data:
    changes = datapoint.get('all_code_changes', [])
    has_test = False
    for change in changes:
        filename = change.get('filename', '')
        if is_test_file(filename):
            has_test = True
    if has_test:
        records_with_test += 1
    else:
        records_without_test += 1

total_records = records_with_test + records_without_test

print("=== VFC record level (does the commit touch any test file?) ===")
print(f"  Records with test changes    : {records_with_test:>4}  ({100 * records_with_test / total_records:.1f}%)")
print(f"  Records without test changes : {records_without_test:>4}  ({100 * records_without_test / total_records:.1f}%)")
print(f"  Total records                : {total_records:>4}")


=== VFC record level (does the commit touch any test file?) ===
  Records with test changes    :  100  (52.6%)
  Records without test changes :   90  (47.4%)
  Total records                :  190
